# De Bag of Words a LLMs: análisis de sentimiento en IMDB

Este notebook corre **todas las etapas** del proyecto y genera las tablas, los gráficos y la sección de resultados del README.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.

| Etapa | Tiempo aproximado en Colab (T4) |
|---|---|
| 1. BoW + Naive Bayes (6 preprocesamientos con validación cruzada) | 15–20 min |
| 2. TF-IDF + regresión logística | 2–3 min |
| 3. Word2Vec + regresión logística | 10–15 min |
| 4a. DistilBERT SST-2 sin ajuste | ~5 min |
| 4b. BERT fine-tuneado | 25–45 min |
| 5. LLM zero-shot y few-shot | 15–30 min |

Los tiempos son estimaciones y dependen de la GPU que asigne Colab.

**Si Colab se desconecta:** las predicciones de cada etapa se guardan en Google Drive apenas terminan. Volvé a correr el notebook desde arriba: las etapas ya hechas se saltean.

In [ ]:
# ── Configuración ─────────────────────────────────────────────────────────────
GUARDAR_EN_DRIVE = True   # recomendado: protege los resultados ante desconexiones
REENTRENAR = False        # True para rehacer etapas que ya tienen predicciones
LLM_MODELO = "Qwen/Qwen2.5-1.5B-Instruct"   # cualquier LLM instruccional de Hugging Face
REPO = "https://github.com/Agustin-Wencelblat/nlp_TPs.git"

In [ ]:
# ── Preparación del entorno ───────────────────────────────────────────────────
import os, sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not os.path.exists("/content/nlp_TPs"):
        !git clone -q {REPO} /content/nlp_TPs
    os.chdir("/content/nlp_TPs")
    !pip install -q -r requirements.txt
    if GUARDAR_EN_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["NLP_RESULTS_DIR"] = "/content/drive/MyDrive/nlp_TPs_results"
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import nltk
for recurso in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng", "averaged_perceptron_tagger"]:
    nltk.download(recurso, quiet=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU (cambiá el entorno de ejecución)")

import pandas as pd
from IPython.display import Image, display
from src.config import RESULTS_DIR, FIG_DIR
from src.correr import correr
print("Resultados en:", RESULTS_DIR)

## Etapa 1 · Bag of Words + Naive Bayes

Representación por conteo de palabras y un clasificador probabilístico simple. Se comparan seis preprocesamientos con **validación cruzada de 5 folds sobre train** y se elige el mejor con ese criterio. El test set no participa de la elección.

Dos correcciones respecto de la primera versión: la lista de stopwords de NLTK incluye *not*, *no* y *nor* (sacarlas convierte "not good" en "good"), y el lematizador ahora recibe la etiqueta gramatical de cada palabra.

In [ ]:
correr(["bow"], REENTRENAR)
pd.read_csv(RESULTS_DIR / "bow_variantes.csv")

## Etapa 2 · TF-IDF + regresión logística

Pondera cada término según cuán informativo es y agrega bigramas, que capturan expresiones como *not good* o *the worst*.

In [ ]:
correr(["tfidf"], REENTRENAR)
pd.read_csv(RESULTS_DIR / "tfidf_top_atributos.csv")

## Etapa 3 · Word2Vec + regresión logística

Vectores densos aprendidos sobre las reseñas de train y las 50.000 sin etiqueta del split *unsupervised*. Cada reseña se representa con el promedio de los vectores de sus palabras. Los vecinos más cercanos muestran una limitación: palabras opuestas como *good* y *bad* aparecen en contextos parecidos y quedan cerca.

In [ ]:
correr(["w2v"], REENTRENAR)
pd.read_csv(RESULTS_DIR / "w2v_vecinos.csv").groupby("palabra")["vecino"].apply(", ".join)

## Etapa 4a · DistilBERT entrenado en SST-2, sin ajuste a IMDB

Un clasificador de sentimiento ya entrenado en otro dataset (frases cortas de reseñas, SST-2) aplicado directamente a IMDB. Mide cuánto se transfiere a otro dominio sin entrenar nada.

In [ ]:
correr(["sst2"], REENTRENAR)

## Etapa 4b · BERT fine-tuneado

`bert-base-uncased` ajustado 2 épocas sobre IMDB, con precisión mixta y padding dinámico. Es la etapa más larga.

In [ ]:
correr(["bert"], REENTRENAR)

## Etapa 5 · LLM zero-shot y few-shot

Un LLM instruccional abierto, sin entrenamiento. En lugar de generar texto y buscar la palabra en la respuesta, se comparan directamente las probabilidades que el modelo asigna a *positive* y *negative* como primera palabra. Eso da una probabilidad por reseña y elimina las respuestas imposibles de interpretar.

El few-shot usa 4 ejemplos balanceados (2 por clase) elegidos al azar de train. Por costo, se evalúa sobre una **submuestra fija de 1.000 reseñas del test**, la misma sobre la que se comparan todos los modelos.

In [ ]:
correr(["llm"], REENTRENAR, llm_modelo=LLM_MODELO)

## Evaluación

Intervalos de confianza por bootstrap, tests de McNemar, calibración, challenge set y costo. Esta celda también actualiza la sección de resultados del `README.md`.

In [ ]:
from src.analizar import main
_ = main()
for nombre in ["accuracy", "calibracion", "challenge", "costo"]:
    display(Image(filename=str(FIG_DIR / f"{nombre}.png")))

## Descargar los resultados

Genera `nlp_TPs_resultados.zip` con la carpeta `results/` y el `README.md` actualizado. Descomprimilo en la raíz del repo (reemplazando lo que haya) y hacé commit.

In [ ]:
import shutil, tempfile
from pathlib import Path

paquete = Path(tempfile.mkdtemp())
shutil.copytree(RESULTS_DIR, paquete / "results")
shutil.copy("README.md", paquete / "README.md")
zip_path = shutil.make_archive("nlp_TPs_resultados", "zip", paquete)
print("Listo:", zip_path)
if EN_COLAB:
    from google.colab import files
    files.download(zip_path)